In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

In [4]:
OUT = Path("data/output")
OUT.mkdir(parents=True, exist_ok=True)

In [5]:
#Generate master data PRODUCTS REGIONS ACCOUNTS and dates
PRODUCTS = pd.DataFrame({
    "product_id" : ["P01", "P02", "P03", "P04"],
    "product_name" : ["Alpha", "Beta", "Gamma", "Delta"],
    "base_price" : [100, 120, 200, 300],
})
REGIONS = pd.DataFrame({
    "region_id" : ["NA", "EU", "APAC", "OTHER"],
    "region_name" : ["North America", "Europe", "Asia, Pacific", "Other"],
})
ACCOUNTS = pd.DataFrame({
    "account_id" : ["REV", "COGS", "PAYROLL", "MARKETING", "IT"],
    "account_name" : ["Revenue", "COGS", "Payroll", "Marketing", "IT"],
    "account_type" : ["Revenue", "COGS", "OPEX", "OPEX", "OPEX"]
})

def _dates():
    return pd.date_range("2024-01-01", "2026-12-31", freq = "MS")


In [6]:
#Generate business scenarios
def _scenario_factor(date, product, region, rng):
    dates = _dates()
    volume = 1.0
    price = 1.0
    cost = 1.0
    if(pd.Timestamp("2026-04-01") <= date and pd.Timestamp("2026-06-01") >= date ):
        if("product_id" == "P01" and "region_id" == "EU"):
            volume *= 0.85
        if("product_id" == "P02" and "region_id" == "NA"):
            price *= 1.06
        cost *= 1.07
    if(pd.Timestamp("2026-05-01") <= date and "region_id" == "EU"):
        volume *= 0.97
    return volume, price, cost

In [15]:
def generate_all(seed = 42):
    rng = np.random.default_rng(seed)
    dates = _dates()
    
    #Export master data
    PRODUCTS.to_csv(OUT/"dim_products.csv", index=False)
    REGIONS.to_csv(OUT/"dim_regions.csv", index = False)
    ACCOUNTS.to_csv(OUT/"dim_accounts.csv", index = False)
    
    #Generate 100 random customer number and assign them randomly to each region with weight
    CUSTOMERS = pd.DataFrame({
        "customer_id" : [f"C{i:03d}" for i in range(1, 101)],
        "region_id" : rng.choice(REGIONS.region_id, 100, p = [0.35, 0.30, 0.25, 0.10]),
    })
    
    #Export master data
    CUSTOMERS.to_csv(OUT/"dim_customers.csv", index = False)
    
    #Generate transaction data - Actuals
    ACTUALS = []
    
    #Create records in each date a product_id and a region_id, calculate the volume, price, dicount, revenue
    for date in dates:
        season = 1 + 0.08 * np.sin(2*np.pi*(date.month-1)/12) #Add seasonality to volume
        for _, p in PRODUCTS.iterrows():
            for region in REGIONS.region_id:
                base = rng.uniform(500, 1800)
                volume = base * season * (1+rng.normal(0, 0.06)) #Add randomization to volume
                volume_factor, price_factor, cost_factor = _scenario_factor(date, p.product_id, region, rng)
                volume *= volume_factor 
                price = p.base_price * price_factor * (1+rng.normal(0, 0.025))
                discount = np.clip(rng.normal(0.06, 0.02), 0, 0.15)
                revenue = volume * price * (1-discount)
                ACTUALS.append([date, p.product_id, region, volume, price, discount, revenue])
    
    SALES = pd.DataFrame(ACTUALS, columns = ["date", "product_id", "region_id", "volume", "price", "discount", "revenue"])
    
    #Generate Budget
    BUDGET = SALES.copy()
    
    BUDGET["volume"] = BUDGET["volume"] / np.where((BUDGET.date >= "2026-04-01") & (BUDGET.product_id == "P01") & (BUDGET.region_id == "EU"), .85, 1)
    BUDGET["price"] = BUDGET["price"] / np.where((BUDGET.date >= "2026-04-01") & (BUDGET.product_id == "P02") & (BUDGET.region_id == "NA"), 1.06, 1)
    BUDGET["revenue"] = BUDGET["volume"] * BUDGET["price"] * (1-BUDGET["discount"])
 
    #Export Sales actuals and budget data
    SALES.to_csv(OUT/"fact_sales_actual.csv", index = False)
    BUDGET.to_csv(OUT/"fact_sales_budget.csv", index = False)

    #Generate Forecast - V1 as 0.25 of Actuals and 0.75 of Budget, V2 as 0.5 of Actuals and 0.5 of Budget, V3 as 0.75 of Actuals and 0.25 of Budget    
    for version, weight in [("v1", 0.25), ("v2", 0.5), ("v3", 0.75)]: 
        F = BUDGET.copy()
        F["volume"] = weight * SALES["volume"] + (1-weight) * BUDGET["volume"]
        F["price"] = weight * SALES["price"] + (1-weight) * BUDGET["price"]
        F["revenue"] = F["volume"] * F["price"] * (1-F["discount"])
        F.to_csv(OUT/f"fact_sales_forecast_{version}.csv", index = False) #Export forecast data
    
        
    #Transform SALES data to financial reporting data
    #Aggregate SALES/revenue by month and region, create monthly Opex
    FINANCE = []
    for date in dates:
        rev_month = SALES[(SALES.date == date)]
        rev_by_region = rev_month.groupby("region_id").revenue.sum()
        for region_id, revenue in rev_by_region.items():
            cogs = -revenue * 0.47
            payroll = -850_000 / 12
            marketing = -220_000 / 12
            it = -110_000 / 12
            travel = -75_000 / 12
            for account, value in [("REV", revenue), ("COGS", cogs), ("PAYROLL", payroll), ("MARKETING", marketing), ("IT", it), ("TRAVEL", travel)]:
                FINANCE.append([date, region_id, account, value])
    
    FINANCE_ACT = pd.DataFrame(FINANCE, columns = ["date", "region_id", "account_id", "value"])
    
    #Generate budget
    FINANCE_BUD = FINANCE_ACT.copy()
    FINANCE_BUD.loc[FINANCE_BUD.account_id == "REV", "value"] *= 1.04
    FINANCE_BUD.loc[FINANCE_BUD.account_id == "COGS", "value"] *= 0.98
    FINANCE_BUD.loc[FINANCE_BUD.account_id.isin(["PAYROLL", "MARKETING", "IT", "TRAVEL"]), "value"] *= 0.97
    
    #Export financial actuals and budget data
    FINANCE_ACT.to_csv(OUT/"fact_finance_actual.csv", index = False)
    FINANCE_BUD.to_csv(OUT/"fact_finance_budget.csv", index = False)
   
    #Generate Forecast - V1 as 0.25 of Actuals and 0.75 of Budget, V2 as 0.5 of Actuals and 0.5 of Budget, V3 as 0.75 of Actuals and 0.25 of Budget
    for version, weight in [("v1", 0.25), ("v2", 0.5), ("v3", 0.75)]:
        F = FINANCE_BUD.copy()
        F["value"] = weight * FINANCE_ACT["value"] + (1-weight) * FINANCE_BUD["value"]
        F.to_csv(OUT/f"fact_finance_forecast_{version}.csv", index = False) #Export forecast data
        
    #Create operational KPI
    #Calculate volume, price, revenue by date, region, and product
    OPERATION = SALES.groupby(["date", "region_id", "product_id"], as_index = False).agg(
    volume = ("volume", "sum"), average_price = ("price", "mean"), revenue = ("revenue", "sum"))
    
    #Add customer count to operation data
    OPERATION["customer_count"] = rng.integers(40, 100, len(OPERATION))
    
    #Export operation data
    OPERATION.to_csv(OUT/"fact_operation.csv", index = False)

In [16]:
generate_all()